In [ ]:
import os
from huggingface_hub import snapshot_download

# Define the repository name and destination path
repo_id = "AI4A-lab/RecruitView"
local_dir = r"data/raw/AI4A-lab/RecruitView"

print(f"Starting download from Hugging Face repository: {repo_id}...")

try:
    # This downloads all repository files into your local project directory
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=local_dir,
        local_dir_use_symlinks=False
    )
    print(f"✅ Dataset successfully downloaded and saved to: {local_dir}")
except Exception as e:
    print(f"❌ Error downloading dataset: {e}")
    print("Please make sure you have the 'huggingface_hub' package installed (pip install huggingface_hub).")

In [ ]:
import os
import glob
import json
import librosa

dataset_dir = r"data/raw/AI4A-lab/RecruitView"
videos_dir = os.path.join(dataset_dir, "videos")
metadata_path = os.path.join(dataset_dir, "metadata.jsonl")

# 1. Scan everything inside the 'videos' folder
video_folder_files = glob.glob(os.path.join(videos_dir, "**/*.*"), recursive=True) if os.path.exists(videos_dir) else []
print(f"Total files found inside 'videos' folder: {len(video_folder_files)}")

if len(video_folder_files) > 0:
    print("Sample files inside 'videos':", [os.path.basename(f) for f in video_folder_files[:3]])

# 2. Read the first few lines of metadata.jsonl to see how transcripts are structured
print(f"\nChecking metadata file: {metadata_path}")
if os.path.exists(metadata_path):
    with open(metadata_path, 'r', encoding='utf-8') as f:
        print("First line of metadata.jsonl:")
        first_line = json.loads(f.readline())
        print(json.dumps(first_line, indent=2))
else:
    print("❌ metadata.jsonl not found!")

# 3. Try to load the first file using librosa (librosa can automatically extract audio from .mp4 videos too!)
if len(video_folder_files) > 0:
    sample_file_path = video_folder_files[0]
    try:
        y, sr = librosa.load(sample_file_path, sr=None)
        print(f"\n✅ Successfully loaded audio from: {os.path.basename(sample_file_path)}")
        print(f"Sampling Rate: {sr} Hz | Duration: {librosa.get_duration(y=y, sr=sr):.2f} seconds")
    except Exception as e:
        print(f"❌ Librosa failed to load this file format directy: {e}")

In [ ]:
import os
import json
import librosa
import numpy as np
from moviepy.video.io.VideoFileClip import VideoFileClip

dataset_dir = r"data/raw/AI4A-lab/RecruitView"
metadata_path = os.path.join(dataset_dir, "metadata.jsonl")

# 1. Load sample entry from JSONL metadata
with open(metadata_path, 'r', encoding='utf-8') as f:
    sample_entry = json.loads(f.readline())

# Get values from metadata dictionary
file_rel_path = sample_entry["file_name"]
transcript_text = sample_entry["transcript"]
print(f"📄 Target Transcript Loaded:\n{transcript_text[:120]}...\n")

# 2. Build absolute path to the video file
video_abs_path = os.path.join(dataset_dir, file_rel_path.replace("/", os.sep))
temp_audio_path = "temp_audio.wav"

# 3. Extract Audio directly from MP4 using explicit moviepy classes
print(f"🎬 Processing Video: {video_abs_path}")
video = VideoFileClip(video_abs_path)
if video.audio is not None:
    video.audio.write_audiofile(temp_audio_path, logger=None)
video.close()

# 4. Load the extracted audio array into Librosa
if os.path.exists(temp_audio_path):
    y, sr = librosa.load(temp_audio_path, sr=None)
    print(f"\n✅ Audio Matrix Loaded Successfully via Librosa!")
    print(f"Sampling Rate: {sr} Hz | Total Signal Samples: {len(y)}")
    
    # Clean up temp file from directory
    os.remove(temp_audio_path)
else:
    print("❌ Error: Audio extraction failed.")

In [ ]:
# ==========================================
# FEATURE EXTRACTION ENGINE (Librosa Based)
# ==========================================

# 1. MFCC Extraction (features/mfcc.py logic)
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
mfccs_mean = np.mean(mfccs, axis=1)
print(f"✅ MFCC Extraction Done. Coefficients shape: {mfccs_mean.shape}")

# 2. Pitch Tracking (features/pitch.py logic)
pitches, voiced_flag, voiced_probs = librosa.pyin(
    y, 
    fmin=librosa.note_to_hz('C2'), 
    fmax=librosa.note_to_hz('C7')
)
mean_pitch = np.nanmean(pitches) if np.any(~np.isnan(pitches)) else 0.0
print(f"✅ Mean Pitch (F0): {mean_pitch:.2f} Hz")

# 3. Energy/Volume Analysis (features/energy.py logic)
rms_energy = librosa.feature.rms(y=y)
mean_energy = np.mean(rms_energy)
print(f"✅ Mean RMS Energy: {mean_energy:.4f}")

# 4. Zero Crossing Rate (features/zcr.py logic)
zcr = librosa.feature.zero_crossing_rate(y)
mean_zcr = np.mean(zcr)
print(f"✅ Mean Zero Crossing Rate: {mean_zcr:.4f}")

# 5. Speech Rate & Silence Statistics (features/speech_rate.py & pause_silence.py logic)
duration = librosa.get_duration(y=y, sr=sr)
total_words = len(transcript_text.split())  # Dynamic word count from metadata!
speech_rate_wpm = (total_words / duration) * 60

# Silence tracking (30 dB cutoff threshold)
intervals = librosa.effects.split(y, top_db=30)
total_speaking_time = sum([end - start for start, end in intervals]) / sr
total_silence_time = max(0.0, duration - total_speaking_time)

print(f"✅ Duration: {duration:.2f}s | Words: {total_words} | Speech Rate: {speech_rate_wpm:.2f} WPM")
print(f"✅ Total Pause/Silence Duration: {total_silence_time:.2f}s")

In [ ]:
import os
import json
import librosa
import numpy as np
import pandas as pd
from moviepy.video.io.VideoFileClip import VideoFileClip
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import threading

dataset_dir = r"data/raw/AI4A-lab/RecruitView"
metadata_path = os.path.join(dataset_dir, "metadata.jsonl")

# Thread-safe lock for printing or accessing shared files if needed
print_lock = threading.Lock()

def process_single_file_threaded(record):
    file_rel_path = record.get("file_name")
    transcript_text = record.get("transcript", "")
    video_abs_path = os.path.join(dataset_dir, file_rel_path.replace("/", os.sep))
    
    # Thread-specific unique temporary audio file to avoid overlap crashes
    thread_id = threading.get_ident()
    temp_audio_path = f"temp_audio_thread_{thread_id}_{record.get('id')}.wav"
    
    if not os.path.exists(video_abs_path):
        return None
        
    try:
        # Extract audio stream using explicit MoviePy class
        video = VideoFileClip(video_abs_path)
        if video.audio is not None:
            video.audio.write_audiofile(temp_audio_path, logger=None)
        video.close()
        
        if not os.path.exists(temp_audio_path):
            return None
            
        # Load audio data matrix into Librosa
        y, sr = librosa.load(temp_audio_path, sr=None)
        
        # Clean up disk space immediately
        if os.path.exists(temp_audio_path):
            os.remove(temp_audio_path)
        
        # --- FEATURE COMPUTATION ENGINE ---
        # 1. MFCC
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs, axis=1)
        
        # 2. Optimized Pitch Tracking
        pitches, voiced_flag, voiced_probs = librosa.pyin(
            y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), hop_length=2048
        )
        mean_pitch = np.nanmean(pitches) if np.any(~np.isnan(pitches)) else 0.0
        
        # 3. Energy
        rms_energy = librosa.feature.rms(y=y)
        mean_energy = np.mean(rms_energy)
        
        # 4. Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y)
        mean_zcr = np.mean(zcr)
        
        # 5. Speech Stats
        duration = librosa.get_duration(y=y, sr=sr)
        total_words = len(transcript_text.split())
        speech_rate_wpm = (total_words / duration) * 60 if duration > 0 else 0.0
        
        intervals = librosa.effects.split(y, top_db=30)
        total_speaking_time = sum([end - start for start, end in intervals]) / sr
        total_silence_time = max(0.0, duration - total_speaking_time)
        
        # Map output dictionary row
        feature_row = {
            "id": record.get("id"),
            "file_name": os.path.basename(video_abs_path),
            "duration_label": record.get("duration"),
            "question_id": record.get("question_id"),
            "question": record.get("question"),
            "openness": record.get("openness"),
            "conscientiousness": record.get("conscientiousness"),
            "extraversion": record.get("extraversion"),
            "agreeableness": record.get("agreeableness"),
            "neuroticism": record.get("neuroticism"),
            "overall_personality": record.get("overall_personality"),
            "interview_score": record.get("interview_score"),
            "Duration_Sec": round(duration, 2),
            "Speech_Rate_WPM": round(speech_rate_wpm, 2),
            "Silence_Duration_Sec": round(total_silence_time, 2),
            "Mean_Pitch_Hz": round(mean_pitch, 2),
            "Mean_Energy": round(mean_energy, 4),
            "Mean_ZCR": round(mean_zcr, 4),
            "transcript": transcript_text
        }
        
        for i, mfcc_val in enumerate(mfccs_mean):
            feature_row[f"MFCC_{i+1}"] = round(mfcc_val, 4)
            
        return feature_row
        
    except Exception as e:
        if os.path.exists(temp_audio_path):
            os.remove(temp_audio_path)
        return None

# Execution Block
if __name__ == "__main__":
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata_lines = [json.loads(line) for line in f]

    print(f"Loaded {len(metadata_lines)} records. Launching thread workers...")
    all_features_data = []

    # Using 4 threads is ideal on Windows to avoid disk bottlenecks during audio extraction
    num_threads = 4
    print(f"Running pipeline using {num_threads} concurrent execution threads.")

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(process_single_file_threaded, rec): rec for rec in metadata_lines}
        
        for future in tqdm(as_completed(futures), total=len(futures)):
            result = future.result()
            if result is not None:
                all_features_data.append(result)

    # Compile and save
    df_final = pd.DataFrame(all_features_data)
    output_dir = "data"
    os.makedirs(output_dir, exist_ok=True)
    output_csv_file = os.path.join(output_dir, "recruitview_final_extracted_features.csv")

    df_final.to_csv(output_csv_file, index=False)
    print(f"\n Complete! Extracted feature matrix saved to: {output_csv_file}")